# Training & Inference su Colab (4 canali)
Pipeline con tiles 224x224 (RGB+mask) preprocessati su Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, random, subprocess
from pathlib import Path
import numpy as np
import torch

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

REPO_DIR = Path('/content/drive/MyDrive/AN2DL_Challenge2-TheBigBatchTheory/an2dl-challenges-25-26')
if not REPO_DIR.exists():
    subprocess.run(['git','clone','https://github.com/asarraa/an2dl-challenges-25-26', str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR / 'challenge2'))

DATA_VARIANT = 'tiny_224_imagenet'  # tiles 224x224 senza npy
DATA_ROOT = Path('/content/drive/MyDrive/AN2DL_Challenge2-TheBigBatchTheory/data/dataset/temp_processed')
BASE_PATH = DATA_ROOT / DATA_VARIANT
TRAIN_IMG_DIR = BASE_PATH / 'train' / 'images'
TRAIN_MSK_DIR = BASE_PATH / 'train' / 'masks'
TEST_IMG_DIR = BASE_PATH / 'test' / 'images'
TEST_MSK_DIR = BASE_PATH / 'test' / 'masks'
CSV_PATH = BASE_PATH / 'train_patches.csv'
assert TRAIN_IMG_DIR.exists(), f'Images dir not found: {TRAIN_IMG_DIR}'
assert TRAIN_MSK_DIR.exists(), f'Masks dir not found: {TRAIN_MSK_DIR}'
assert CSV_PATH.exists(), f'CSV not found: {CSV_PATH}'
print(f'Using device: {device}')
print(f'Dataset base: {BASE_PATH}')


In [ ]:
!pip install -q comet_ml torchsummary


## Dataloaders (4 canali con mask)

In [ ]:
import pandas as pd
import cv2
import torch.utils.data as torch_data
from sklearn.model_selection import train_test_split
from torchvision.transforms import v2 as transforms
from lazy_loaders import LazyImageDataset, LOADER_PARAMS, ColorJitterRGB

BATCH_SIZE = 128
ADD_MASK_CHANNEL = True
df = pd.read_csv(CSV_PATH)
label_map = {label: idx for idx, label in enumerate(df['label'].unique())}
inv_label_map = {v:k for k,v in label_map.items()}
df['label'] = df['label'].map(label_map)

train_df, val_df = train_test_split(
    df,
    test_size=LOADER_PARAMS['percentage_validation'],
    random_state=SEED,
    stratify=df['label']
)

train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    ColorJitterRGB(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1))
])

train_ds = LazyImageDataset(train_df, TRAIN_IMG_DIR, masks_dir=TRAIN_MSK_DIR, add_mask_channel=ADD_MASK_CHANNEL, transform=train_aug)
val_ds = LazyImageDataset(val_df, TRAIN_IMG_DIR, masks_dir=TRAIN_MSK_DIR, add_mask_channel=ADD_MASK_CHANNEL, transform=None)

coverage = train_df['tumor_coverage'].to_numpy()
weights = coverage + 1e-3
train_sampler = torch_data.WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

num_workers = max(2, min(8, os.cpu_count() or 2))
train_loader = torch_data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
    num_workers=num_workers, pin_memory=True, pin_memory_device='cuda' if torch.cuda.is_available() else '',
    prefetch_factor=4, persistent_workers=True, drop_last=False
)
val_loader = torch_data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    num_workers=num_workers, pin_memory=True, pin_memory_device='cuda' if torch.cuda.is_available() else '',
    prefetch_factor=4, persistent_workers=True
)

sample_img = cv2.imread(str(TRAIN_IMG_DIR / train_df.iloc[0]['sample_index']))
sample_mask = cv2.imread(str(TRAIN_MSK_DIR / train_df.iloc[0]['sample_index']), cv2.IMREAD_GRAYSCALE)
if sample_mask is None:
    raise FileNotFoundError(f"Mask missing for {train_df.iloc[0]['sample_index']}")
input_shape = (4, sample_img.shape[0], sample_img.shape[1])
print(f'Train samples: {len(train_ds)}, Val samples: {len(val_ds)}')
print(f'Input shape: {input_shape}')


## Training (CNN 4 canali)

In [ ]:
from launch_training import start_training

TRAINING_PARAMS = {
    'epochs': 100,
    'learning_rate': 1e-3,
    'patience': 20,
}
MODEL_PARAMS = {
    'input_shape': input_shape,
}

trained_model, history = start_training(
    model_name='CNN',
    model_params=MODEL_PARAMS,
    training_params=TRAINING_PARAMS,
    device=device,
    train_loader=train_loader,
    val_loader=val_loader,
    data_input_shape=input_shape,
)


## Inference sul test set (4 canali)

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import v2 as transforms

class InferenceDataset(Dataset):
    def __init__(self, images_dir, masks_dir=None, add_mask_channel=True, transform=None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir else None
        self.add_mask_channel = add_mask_channel
        self.transform = transform
        self.files = sorted([p.name for p in self.images_dir.glob('*.png')])
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        name = self.files[idx]
        img = cv2.imread(str(self.images_dir / name))
        if img is None:
            raise FileNotFoundError(f'Image missing: {name}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.add_mask_channel:
            mask = cv2.imread(str(self.masks_dir / name), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise FileNotFoundError(f'Mask missing: {name}')
            if mask.shape[:2] != img.shape[:2]:
                mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
            img = np.dstack((img, mask))
        img = Image.fromarray(img)
        img = transforms.ToImage()(img)
        img = transforms.ToDtype(torch.float32, scale=True)(img)
        if self.transform:
            img = self.transform(img)
        return img, name

test_ds = InferenceDataset(TEST_IMG_DIR, masks_dir=TEST_MSK_DIR, add_mask_channel=True, transform=None)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers,
    pin_memory=True, pin_memory_device='cuda' if torch.cuda.is_available() else '',
    prefetch_factor=4, persistent_workers=True
)

trained_model.eval()
preds = []
names = []
with torch.no_grad():
    for imgs, nms in test_loader:
        imgs = imgs.to(device)
        logits = trained_model(imgs)
        probs = F.softmax(logits, dim=1)
        labels = probs.argmax(dim=1).cpu().numpy()
        names.extend(nms)
        preds.extend(labels)

pred_labels = [inv_label_map[i] for i in preds]
submission = pd.DataFrame({'sample_index': names, 'label': pred_labels})
out_path = BASE_PATH / 'test_predictions.csv'
submission.to_csv(out_path, index=False)
print(submission.head())
print(f'Saved predictions to {out_path}')
